# Multilingual Voice-Based Government Scheme Recommendation Assistant
**Demo notebook** - runs the Streamlit app in Colab through ngrok.

**Before you start**
1. `Runtime -> Change runtime type -> T4 GPU` (Whisper is much faster on GPU).
2. Have your ngrok authtoken ready (dashboard.ngrok.com -> Your Authtoken).
3. `processed_schemes.pkl` and `scheme_tfidf.pkl` must be in Google Drive `MyDrive` (or already in `/content`).

**Run the cells from top to bottom.** After the last cell prints the link, open it in **Chrome**, click *Visit Site* on the ngrok page, allow the microphone, and wait for the models to finish loading (first time only).

In [ ]:
# 1. Install everything (same libraries as the project notebook, plus Streamlit + pyngrok)
!pip install -q openai-whisper gTTS transformers sentencepiece "click<8.2" "streamlit>=1.40" pyngrok

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 1.16.1 which is incompatible.
spacy 3.8.18 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.


In [ ]:
# 2. Scheme data: copy the two saved files from Google Drive into /content
import os, shutil
from google.colab import drive

drive.mount('/content/drive')

for name in ["processed_schemes.pkl", "scheme_tfidf.pkl"]:
    src = f"/content/drive/MyDrive/{name}"
    dst = f"/content/{name}"
    if os.path.exists(dst):
        print(name, "-> already in /content")
    elif os.path.exists(src):
        shutil.copy(src, dst)
        print(name, "-> copied from Google Drive")
    else:
        print("MISSING:", name, "- put it in Google Drive (MyDrive) or upload it to /content")

Mounted at /content/drive
processed_schemes.pkl -> copied from Google Drive
scheme_tfidf.pkl -> copied from Google Drive


### Project modules
These five cells are your existing, tested files, unchanged (`voice_format`, `speech_input`, `speech_output`, `translate`, `text_cleaning`).

In [ ]:
%%writefile voice_format.py

def first_sentence(text):
    if not text:
        return ""

    text = str(text).strip()

    # Protect common abbreviations
    text = text.replace("Rs.", "Rs")

    # Take the first sentence
    parts = text.split(".")

    sentence = parts[0].strip()

    # Restore Rs.
    sentence = sentence.replace("Rs", "Rs.")

    return sentence + "."


def format_recommendations(results, top_n=3):

    results = results.head(top_n)

    response = "I found some government schemes that may help you.\n\n"

    for i, (_, row) in enumerate(results.iterrows(), start=1):

        response += f"{i}. {row['scheme_name']}\n"

        response += f"Benefit: {first_sentence(row['benefits'])}\n"

        response += f"Eligibility: {first_sentence(row['eligibility'])}\n\n"

    return response
def to_speakable(text):
    return text.replace("\n", " ").strip()

Writing voice_format.py


In [ ]:
%%writefile speech_input.py

"""
Speech-to-text wrapper around OpenAI Whisper, hardened against the
repetition/looping bug we hit during testing.
"""

import whisper

_model = None


def get_model(size: str = "small"):
    """Load and cache the Whisper model."""
    global _model

    if _model is None:
        _model = whisper.load_model(size)

    return _model


def speech_to_text(
    audio_file: str,
    language: str = "auto",
    model_size: str = "small"
) -> str:

    model = get_model(model_size)

    result = model.transcribe(
        audio_file,
        language=None if language == "auto" else language,
        condition_on_previous_text=False,
        temperature=(0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
        no_speech_threshold=0.6,
        compression_ratio_threshold=2.0,
    )

    text = result.get("text", "").strip()

    return text

Writing speech_input.py


In [ ]:
%%writefile speech_output.py

from gtts import gTTS


def text_to_speech(text, language="en", output_file="output.mp3"):
    tts = gTTS(text=text, lang=language)
    tts.save(output_file)

Writing speech_output.py


In [ ]:
%%writefile translate.py

"""
Translation between English and Hindi/Tamil.

Model choices:
  en -> hi : facebook/nllb-200-distilled-600M
  hi -> en : facebook/nllb-200-distilled-600M
  en -> ta : gopi30/english-to-tamil-stage4
  ta -> en : Helsinki-NLP/opus-mt-dra-en

NLLB is used for both Hindi directions because the previous
MarianMT models produced incorrect or repetitive translations
on some scheme-related text.

A validation + fallback check is used on every translation:
if the output looks broken, the original text is returned.
"""

import re

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    M2M100ForConditionalGeneration,
    M2M100Tokenizer,
)


# NLLB language codes
_NLLB_CODES = {
    "en": "eng_Latn",
    "hi": "hin_Deva"
}


_MODEL_NAMES = {
    ("en", "hi"): ("nllb", "facebook/nllb-200-distilled-600M"),
    ("hi", "en"): ("nllb", "facebook/nllb-200-distilled-600M"),
    ("en", "ta"): ("m2m", "gopi30/english-to-tamil-stage4"),
    ("ta", "en"): ("marian", "Helsinki-NLP/opus-mt-dra-en"),
}


_loaded = {}


def _get_model(src: str, tgt: str):

    key = (src, tgt)

    if key not in _loaded:

        if key not in _MODEL_NAMES:
            raise ValueError(
                f"No translation model configured for {src} -> {tgt}"
            )

        kind, name = _MODEL_NAMES[key]

        tokenizer = (
            AutoTokenizer.from_pretrained(name)
            if kind != "m2m"
            else M2M100Tokenizer.from_pretrained(name)
        )

        model = (
            M2M100ForConditionalGeneration.from_pretrained(name)
            if kind == "m2m"
            else AutoModelForSeq2SeqLM.from_pretrained(name)
        )

        _loaded[key] = (tokenizer, model, kind)

    return _loaded[key]


def _generate(text: str, src: str, tgt: str, max_length: int) -> str:

    tokenizer, model, kind = _get_model(src, tgt)

    gen_kwargs = dict(
        max_length=max_length,
        num_beams=4,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
    )

    if kind == "nllb":

        tokenizer.src_lang = _NLLB_CODES[src]

        inputs = tokenizer(
            text,
            return_tensors="pt",
            padding=True,
            truncation=True
        )

        gen_kwargs["forced_bos_token_id"] = (
            tokenizer.convert_tokens_to_ids(_NLLB_CODES[tgt])
        )

    else:

        inputs = tokenizer(
            text,
            return_tensors="pt",
            padding=True,
            truncation=True
        )

        if kind == "m2m":
            gen_kwargs["forced_bos_token_id"] = tokenizer.get_lang_id(tgt)

    translated_tokens = model.generate(
        **inputs,
        **gen_kwargs
    )

    result = tokenizer.decode(
        translated_tokens[0],
        skip_special_tokens=True
    )

    return result.replace("__ta__", "").strip()


def _looks_broken(original: str, translated: str) -> bool:

    if not translated.strip():
        return True

    tokens = translated.split()

    for i in range(len(tokens) - 2):

        if tokens[i] == tokens[i + 1] == tokens[i + 2]:
            return True

    letters = re.sub(r"[^\w\s]", "", translated)

    if len(original) > 20 and len(letters.strip()) < len(original) * 0.2:
        return True

    return False


def translate(
    text: str,
    src: str,
    tgt: str,
    max_length: int = 256
) -> str:

    if src == tgt or not text.strip():
        return text

    result = _generate(
        text,
        src,
        tgt,
        max_length
    )

    if _looks_broken(text, result):
        return text

    return result


def translate_sentences(
    text: str,
    src: str,
    tgt: str
) -> str:

    if src == tgt:
        return text

    clauses = [
        c.strip()
        for c in re.split(r"[.;:]", text)
        if c.strip()
    ]

    translated = [
        translate(c, src, tgt)
        for c in clauses
    ]

    return ". ".join(translated) + ("." if translated else "")

Writing translate.py


In [ ]:
%%writefile text_cleaning.py

import re

def normalize_for_translation(text):
    text = text.replace("₹", " rupees ")
    text = text.replace("/-", "")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

Writing text_cleaning.py


### The app
`app.py` is only the user interface. It reuses the modules above and your `recommend_schemes` function.

In [ ]:
%%writefile app.py
"""
Multilingual Voice-Based Government Scheme Recommendation Assistant
Streamlit front end (runs in Google Colab through pyngrok).

This file only adds the user interface. All the tested logic is reused:
    speech_input.py    Whisper speech-to-text
    translate.py       English <-> Hindi / Tamil translation
    text_cleaning.py   cleaning before translation
    voice_format.py    response formatting
    speech_output.py   gTTS text-to-speech
    recommend_schemes  70% TF-IDF similarity + 30% profile match (unchanged)

The pipeline steps run in the same order as engine.run_pipeline(); they are
only split into stages so the screen can show "What I heard" and the schemes
before the (optional) audio is generated.
"""

import hashlib
import os
import pickle
import tempfile
import traceback

import joblib
import numpy as np
import streamlit as st
from sklearn.metrics.pairwise import cosine_similarity

st.set_page_config(
    page_title="Government Scheme Assistant",
    page_icon="🎤",
    layout="centered",
)

# ---------------------------------------------------------------------------
# One-time loading (cached, so it does not repeat on every click)
# ---------------------------------------------------------------------------

DATA_DIRS = ["/content", "/content/drive/MyDrive"]


def _find_file(name):
    for folder in DATA_DIRS:
        path = os.path.join(folder, name)
        if os.path.exists(path):
            return path
    raise FileNotFoundError(
        f"{name} not found in {DATA_DIRS}. "
        "Copy it to /content or mount Google Drive."
    )


@st.cache_resource(show_spinner="Loading scheme data...")
def load_scheme_data():
    with open(_find_file("processed_schemes.pkl"), "rb") as f:
        df = pickle.load(f)
    tfidf = joblib.load(_find_file("scheme_tfidf.pkl"))
    X = tfidf.transform(df["scheme_text"])
    return df, tfidf, X


@st.cache_resource(show_spinner=False)
def load_models():
    """Load Whisper and all translation models once, before the demo starts."""
    import speech_input
    import translate as translate_module

    speech_input.get_model("small")

    # Hindi uses the same NLLB model in both directions. Load it once and
    # reuse it for hi -> en (same weights, same output, half the memory).
    translate_module._get_model("en", "hi")
    translate_module._loaded[("hi", "en")] = translate_module._loaded[("en", "hi")]

    translate_module._get_model("en", "ta")
    translate_module._get_model("ta", "en")
    return True


df, tfidf, X = load_scheme_data()

from speech_input import speech_to_text
from speech_output import text_to_speech
from text_cleaning import normalize_for_translation
from translate import translate, translate_sentences
from voice_format import format_recommendations, to_speakable


# ---------------------------------------------------------------------------
# Recommendation engine - the same function as in the notebook, unchanged
# ---------------------------------------------------------------------------

def recommend_schemes(query, user_profile, top_k=5):

    # 1. Convert query to TF-IDF
    query_vector = tfidf.transform([query])

    # 2. Calculate text similarity
    similarity_scores = cosine_similarity(
        query_vector,
        X
    ).flatten()

    # 3. Calculate category scores
    category_scores = []

    for idx in range(len(df)):
        score = 0

        for category, user_value in user_profile.items():
            if user_value and df.iloc[idx][category]:
                score += 1

        category_scores.append(score)

    # 4. Convert to NumPy array
    category_scores = np.array(category_scores)

    # 5. Normalize category scores
    max_category_score = sum(user_profile.values())

    if max_category_score > 0:
        normalized_category_scores = (
            category_scores / max_category_score
        )
    else:
        normalized_category_scores = np.zeros(len(df))

    # 6. Combine scores: 70% text + 30% category
    combined_scores = (
        0.7 * similarity_scores +
        0.3 * normalized_category_scores
    )

    # 7. Get top K schemes
    top_indices = combined_scores.argsort()[-top_k:][::-1]

    # 8. Prepare results
    results = df.iloc[top_indices][
        [
            "scheme_name",
            "benefits",
            "eligibility",
            "tags"
        ]
    ].copy()

    results["similarity_score"] = similarity_scores[top_indices]
    results["category_score"] = category_scores[top_indices]
    results["combined_score"] = combined_scores[top_indices]

    return results


# ---------------------------------------------------------------------------
# Settings shown on the screen
# ---------------------------------------------------------------------------

LANGUAGES = {"English": "en", "Hindi": "hi", "Tamil": "ta"}

# label on screen -> boolean column in the scheme dataset
PROFILE_OPTIONS = {
    "Farmer": "farmer",
    "Student": "student",
    "Senior citizen": "senior_citizen",
    "Woman": "women",
    "Person with disability": "disability",
    "Low income": "low_income",
    "Unemployed": "unemployed",
    "Involved in agriculture": "agriculture",
}

EXAMPLES = {
    "en": "I am a farmer and I need financial assistance for seeds.",
    "hi": "मैं एक किसान हूं और मुझे बीज के लिए आर्थिक सहायता चाहिए।",
    "ta": "எனக்கு அரசு திட்டங்களைப் பற்றி தகவல் வேண்டும்.",
}


# ---------------------------------------------------------------------------
# Pipeline stages (same order as engine.run_pipeline)
# ---------------------------------------------------------------------------

def process_recording(audio_bytes, language, user_profile, status):
    # The recording from the microphone is saved as a .wav file for Whisper
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:
        f.write(audio_bytes)
        audio_path = f.name

    try:
        # 1. Voice -> text
        status.write("Whisper is listening to your recording...")
        heard = speech_to_text(audio_path, language=language)
    finally:
        try:
            os.remove(audio_path)
        except OSError:
            pass

    if not heard.strip():
        return {"heard": ""}

    # 2. Translate to English for the recommendation engine
    status.write("Understanding your question...")
    query_en = translate(heard, src=language, tgt="en")

    # 3. Recommendation engine
    status.write("Finding matching schemes...")
    results = recommend_schemes(query_en, user_profile, top_k=5)

    # 4. Format + speakable text
    status.write("Preparing the explanation...")
    formatted = format_recommendations(results)
    speakable_en = to_speakable(formatted)

    # 5. Clean the response before translation
    speakable_en_clean = normalize_for_translation(speakable_en)

    # 6. Translate back to the user's language
    if language == "en":
        explanation = speakable_en_clean
    else:
        explanation = translate_sentences(
            speakable_en_clean,
            src="en",
            tgt=language,
        )

    return {
        "heard": heard,
        "query_en": query_en,
        "results": results.reset_index(drop=True),
        "explanation": explanation,
    }


def make_voice(text, language):
    # 7. Text -> speech (gTTS), returned as mp3 bytes for the audio player
    out_path = os.path.join(tempfile.gettempdir(), f"response_{language}.mp3")
    text_to_speech(text, language=language, output_file=out_path)
    with open(out_path, "rb") as f:
        return f.read()


# ---------------------------------------------------------------------------
# Screen
# ---------------------------------------------------------------------------

st.markdown("## Multilingual Voice-Based Government Scheme Recommendation Assistant")
st.caption("Speak your question in English, Hindi or Tamil and get scheme recommendations.")

with st.spinner("Loading speech and translation models (first time only)..."):
    load_models()

# Step 1 - language
st.subheader("1. Choose your language")
language_label = st.radio(
    "Language",
    list(LANGUAGES),
    horizontal=True,
    label_visibility="collapsed",
)
language = LANGUAGES[language_label]

# Step 2 - profile
st.subheader("2. Select what applies to you")
left, right = st.columns(2)
selected = []
for i, (label, column) in enumerate(PROFILE_OPTIONS.items()):
    with (left if i < 4 else right):
        if st.checkbox(label, key=f"profile_{column}"):
            selected.append(column)
user_profile = {column: 1 for column in selected}

# Step 3 - voice
st.subheader("3. Ask your question by voice")
st.caption("Choose language and profile first. Then click the microphone, speak, and click stop.")
st.caption(f"Example: {EXAMPLES[language]}")

audio = st.audio_input("Record your question")

if audio is None:
    for key in ("run_key", "output", "voice"):
        st.session_state.pop(key, None)
    st.stop()

audio_bytes = audio.getvalue()
run_key = (
    hashlib.md5(audio_bytes).hexdigest(),
    language,
    tuple(sorted(selected)),
)

# Run the pipeline once per new recording / changed language / changed profile.
# Changing the Text / Voice / Both choice below does NOT run it again.
if st.session_state.get("run_key") != run_key:
    st.session_state.pop("voice", None)
    output = None
    with st.status("Processing your recording...", expanded=True) as status:
        try:
            output = process_recording(audio_bytes, language, user_profile, status)
            status.update(label="Done", state="complete", expanded=False)
        except Exception as error:
            traceback.print_exc()
            status.update(label="Something went wrong", state="error")
            st.error(f"Sorry, something went wrong: {error}")
    st.session_state["run_key"] = run_key
    st.session_state["output"] = output

output = st.session_state.get("output")
if output is None:
    st.stop()

if not output["heard"].strip():
    st.warning("I could not hear anything clearly. Please record again, a little closer to the microphone.")
    st.stop()

# What I heard
st.subheader("What I heard")
st.info(output["heard"])
if language != "en":
    st.caption(f"Understood as (English): {output['query_en']}")

# Recommended schemes
st.subheader("Recommended schemes")
table = output["results"][
    ["scheme_name", "similarity_score", "category_score", "combined_score"]
].copy()
table.columns = ["Scheme", "Text similarity", "Profile match", "Combined score"]
table["Text similarity"] = table["Text similarity"].round(3)
table["Combined score"] = table["Combined score"].round(3)
table.index = range(1, len(table) + 1)
st.table(table)
st.caption("Score = 70% text similarity + 30% profile match. The explanation below covers the top 3.")

# Step 4 - answer format
st.subheader("4. How would you like the explanation?")
mode = st.radio(
    "Explanation format",
    ["Text", "Voice", "Both"],
    index=2,
    horizontal=True,
    label_visibility="collapsed",
)

if mode in ("Text", "Both"):
    st.markdown(output["explanation"].replace("$", "\\$"))

if mode in ("Voice", "Both"):
    if "voice" not in st.session_state:
        with st.spinner("Generating voice..."):
            try:
                st.session_state["voice"] = make_voice(output["explanation"], language)
            except Exception as error:
                traceback.print_exc()
                st.error(f"Could not generate audio (gTTS needs internet): {error}")
    if "voice" in st.session_state:
        st.audio(st.session_state["voice"], format="audio/mp3")


Writing app.py


In [ ]:
!pip install -q pyngrok

In [ ]:
!pip install -q "streamlit>=1.40"

### Start the app

In [ ]:
# 3. Start Streamlit and open the ngrok link
import subprocess, time, urllib.request
from getpass import getpass
from pyngrok import ngrok

NGROK_TOKEN = getpass("Paste your ngrok authtoken: ")

# stop anything left over from an earlier run
ngrok.kill()
subprocess.run(["pkill", "-f", "streamlit run"], check=False)
time.sleep(2)

ngrok.set_auth_token(NGROK_TOKEN)

log = open("streamlit.log", "w")
subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--browser.gatherUsageStats", "false",
    ],
    stdout=log,
    stderr=log,
)

# wait until Streamlit answers
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:8501/_stcore/health", timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    print("Streamlit did not start. Run the log cell below to see why.")

tunnel = ngrok.connect(8501)
print("\nOpen this link in Chrome:", tunnel.public_url)

Paste your ngrok authtoken: ··········

Open this link in Chrome: https://starring-breezy-verbose.ngrok-free.dev


### If something goes wrong
Show the last lines of the Streamlit log (model loading messages and any error appear here).

In [ ]:
!tail -n 40 streamlit.log

Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: app.py


### After the demo

In [ ]:
# Stop the app and close the tunnel
from pyngrok import ngrok
import subprocess

ngrok.kill()
subprocess.run(["pkill", "-f", "streamlit run"], check=False)
print("Stopped.")

In [ ]:
!ls -lh *.py

ls: cannot access '*.py': No such file or directory
